In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import sys

In [4]:
#sys.argv = ['-f'] + ["MIC", "-a", "cpu"]
sys.argv = ['-f'] + ["MIC", "-a", "cpu", "--suppressupdates"]

In [5]:
import os
sys.path.append("/home/pwiesenbach/BertGCN")
os.chdir("/home/pwiesenbach/BertGCN")

In [6]:
from entry import * 

In [7]:
import importlib
import entry
import logging
importlib.reload(logging)
importlib.reload(entry)

<module 'entry' from '/beegfs/homes/pwiesenbach/BertGCN/entry.py'>

In [8]:
import datetime
import json
import pickle
import random
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import Subset
from transformers import AutoTokenizer

from clinic_datasets import CleanClinicDataset
from utils import *
from model import BertGCN
import torch.utils.data as Data
import dgl

import logging

import shap

import dgl
import torch
from captum.attr import IntegratedGradients
from ferret.explainers.gradient import IntegratedGradientExplainer
from functools import partial
from collections import defaultdict
from params import parse_args
from ferret import Benchmark

In [9]:
args = parse_args()
random.seed(0)
np.random.seed(0)
torch.manual_seed(0)

BATCHSIZE = 8

MODELNAME = Path(PRETRAINEDMODEL).stem
if args.data == "MIC":
    DATASET = "med_indication_all_RF_diag"
    DATASETPATH =  Path("data") / f"ind.{DATASET}_{args.doclevel}"
    if args.testunklar:
        DATASETPATH =  Path("data") / f"ind.{DATASET}_{args.doclevel}_testunklar"
    MAXEVALS = 5399
elif args.data == "CSC":
    DATASET = "CARDIODE400_main"
    DATASETPATH =  Path("data") / f"ind.{DATASET}"
    MAXEVALS = 233743

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained(PRETRAINEDMODEL)

if args.data == "MIC":
    dataset_file = Path("data") / f"medindcls_{args.bertmodel}_{args.doclevel}.json"
    if not dataset_file.exists():
        print("Creating dataset")
        dataset = CleanClinicDataset(tokenizer=tokenizer, task="MIC", doclevel=args.doclevel, clean=False)
        with open(dataset_file, "wb") as f:
            print(f"Saving dataset under {dataset_file}")
            pickle.dump(dataset, f)
    else:
        print(f"Loading dataset from: {dataset_file}")
        with open(dataset_file, "rb") as f:
            dataset = pickle.load(f)
elif args.data == "CSC":
    train_dataset_file = Path("data") / "csc_train_bert.json"
    test_dataset_file = Path("data") / "csc_test_bert.json"

    if not train_dataset_file.exists():
        print("Creating train dataset")
        train_dataset = CleanClinicDataset(tokenizer=tokenizer, task="CSC", clean=False, mode="train")
        with open(train_dataset_file, "wb") as f:
            print(f"Saving dataset under {train_dataset_file}")
            pickle.dump(train_dataset, f)
    else:
        print(f"Loading train dataset from: {train_dataset_file}")
        with open(train_dataset_file, "rb") as f:
            train_dataset = pickle.load(f)
    if not test_dataset_file.exists():
        print("Creating test dataset")
        test_dataset = CleanClinicDataset(tokenizer=tokenizer, task="CSC", clean=False, mode="test")
        with open(test_dataset_file, "wb") as f:
            print(f"Saving dataset under {test_dataset_file}")
            pickle.dump(test_dataset, f)
    else:
        print(f"Loading test dataset from: {test_dataset_file}")
        with open(test_dataset_file, "rb") as f:
            test_dataset = pickle.load(f)
    dataset = train_dataset

GCNNAME = f"{MODELNAME}_{dataset}.pt"
SAVEDIR = Path(f"models/gcn/{args.mixfactor}/{args.doclevel}")
GCNPATH= SAVEDIR / GCNNAME

IGFILE = SAVEDIR / f"ig_attrs_gcn_{MODELNAME}_{args.data}"
SHAPFILE = SAVEDIR / f"shap_values_gcn_{MODELNAME}_{args.data}"
if args.testunklar:
    IGFILE = SAVEDIR / f"ig_attrs_gcn_{MODELNAME}_{args.data}_testunklar"
    SHAPFILE = SAVEDIR / f"shap_values_gcn_{MODELNAME}_{args.data}_testunklar"

if args.data == "MIC":
    if not args.testunklar:
        idx = np.arange(len(dataset))
        random.shuffle(idx)
        train_idx, val_idx, test_idx = (
            idx[: int(len(idx) * 0.7)],
            idx[int(len(idx) * 0.7) : int(len(idx) * 0.8)],
            idx[int(len(idx) * 0.8) :],
        )
        train_dataset = Subset(dataset, train_idx)
        val_dataset = Subset(dataset, val_idx)
        test_dataset = Subset(dataset, test_idx)
    else:
        _train_idx, test_idx = list(), list()
        for i, x in enumerate(dataset):
            if "unklar" in dataset.LE.classes_[x["labels"]]:
                test_idx.append(i)
            else:
                _train_idx.append(i)
        test_dataset = Subset(dataset, test_idx)
        random.shuffle(_train_idx)
        train_idx, val_idx = (
            _train_idx[: int(len(_train_idx) * 0.9)],
            _train_idx[int(len(_train_idx) * 0.9) :],
        )
        train_dataset = Subset(dataset, train_idx)
        val_dataset = Subset(dataset, val_idx)
        assert len(train_idx) + len(val_idx) == len(_train_idx)
elif args.data == "CSC":
    idx = np.arange(len(train_dataset))
    random.shuffle(idx)
    train_idx, val_idx = idx[: int(len(idx) * 0.9)], idx[int(len(idx) * 0.9) :]

adj, features, y_train, y_val, y_test, train_mask, val_mask, test_mask, _, _ = load_corpus(DATASETPATH)

nb_node = features.shape[0]
nb_train, nb_val, nb_test = train_mask.sum(), val_mask.sum(), test_mask.sum()
nb_word = nb_node - nb_train - nb_val - nb_test
nb_class = y_train.shape[1]

model = BertGCN(
    nb_class=nb_class,
    pretrained_model=PRETRAINEDMODEL,
    mix_factor=args.mixfactor,
    gcn_layers=2,
    n_hidden=200,
    dropout=0.5,
)

model = model.to(device)
model = model.eval()

y = y_train + y_test + y_val
y_train = y_train.argmax(axis=1)
y = y.argmax(axis=1)

# document mask used for update feature
doc_mask = train_mask + val_mask + test_mask

if args.data == "MIC":
    input_ids = torch.cat(
        [
            torch.tensor(np.array([x["input_ids"] for x in np.array(dataset.examples)[train_idx]])),
            torch.zeros((nb_word, tokenizer.model_max_length), dtype=torch.long),
            torch.tensor(np.array([x["input_ids"] for x in np.array(dataset.examples)[val_idx]])),
            torch.tensor(np.array([x["input_ids"] for x in np.array(dataset.examples)[test_idx]])),
        ]
    )
elif args.data == "CSC":
    input_ids = torch.cat(
        [
            torch.tensor(np.array([x["input_ids"] for x in np.array(dataset.examples)[train_idx]])),
            torch.zeros((nb_word, tokenizer.model_max_length), dtype=torch.long),
            torch.tensor(np.array([x["input_ids"] for x in np.array(dataset.examples)[val_idx]])),
            torch.tensor(np.array([x["input_ids"] for x in test_dataset.examples])),
        ]
    )

assert np.array_equal(y[:nb_train], dataset.labels[train_idx])
assert np.array_equal(y[nb_train + nb_word : nb_train + nb_word + nb_val], dataset.labels[val_idx])
if args.data == "MIC":
    assert np.array_equal(y[-nb_test:], dataset.labels[test_idx])
elif args.data == "CSC":
    assert np.array_equal(y[-nb_test:], test_dataset.labels)

adj_norm = normalize_adj(adj + sp.eye(adj.shape[0]))

train_idx_dataset = Data.TensorDataset(torch.arange(0, nb_train, dtype=torch.long))
val_idx_dataset = Data.TensorDataset(torch.arange(nb_train + nb_word, nb_train + nb_word + nb_val, dtype=torch.long))
test_idx_dataset = Data.TensorDataset(torch.arange(nb_node - nb_test, nb_node, dtype=torch.long))

idx_loader_train = Data.DataLoader(train_idx_dataset, batch_size=BATCHSIZE)
idx_loader_val = Data.DataLoader(val_idx_dataset, batch_size=BATCHSIZE)
idx_loader_test = Data.DataLoader(test_idx_dataset, batch_size=BATCHSIZE)

graph = dgl.from_scipy(adj_norm.astype("float32"), eweight_name="edge_weight")
graph.ndata["input_ids"] = input_ids
graph.ndata["label"], graph.ndata["train"], graph.ndata["val"], graph.ndata["test"] = (
    torch.LongTensor(y),
    torch.FloatTensor(train_mask),
    torch.FloatTensor(val_mask),
    torch.FloatTensor(test_mask),
)
graph.ndata["label_train"] = torch.LongTensor(y_train)
graph.ndata["cls_feats"] = torch.zeros((nb_node, model.feat_dim))

def update_feature():
    global graph, model
    dataloader = Data.DataLoader(Data.TensorDataset(graph.ndata["input_ids"][doc_mask]), batch_size=64)
    with torch.no_grad():
        model.eval()
        cls_list = []
        logging.info("Udating features...")
        for batch in dataloader:
            input_ids = [x.to(device) for x in batch][0]
            output = model.bert_model(input_ids=input_ids)[0][:, 0]
            cls_list.append(output.cpu())
        cls_feat = torch.cat(cls_list, axis=0)
    graph = graph.to("cpu")
    graph.ndata["cls_feats"][doc_mask] = cls_feat

logging.info(f"Loading best gcn model from {GCNPATH} saved on {datetime.datetime.fromtimestamp(GCNPATH.stat().st_ctime)}")
model.load_state_dict(torch.load(GCNPATH, map_location="cpu"))

if not args.suppressupdates:
    update_feature()

Loading dataset from: data/medindcls_medbert_letter.json


Trying to unpickle estimator LabelEncoder from version 0.22 when using version 1.3.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
Trying to unpickle estimator OneHotEncoder from version 0.22 when using version 1.3.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
Some weights of the model checkpoint at /prj/doctoral_letters/PETGUI/med_bert_local were not used when initializing BertModel: ['cls.predictions.bias', 'cls.predictions.transform.dense.weight', 'cls.predictions.decoder.weight', 'cls.predictions.decoder.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.dense.bias']
- This IS expected if you are initializin

In [10]:
IGFILE = SAVEDIR / f"ig_eval_gcn_{MODELNAME}_{args.data}"
if args.testunklar:
    IGFILE = SAVEDIR / f"ig_eval_gcn_{MODELNAME}_{args.data}_testunklar"

with open(f"{IGFILE}.json", "rb") as f:
    evals = pickle.load(f)

In [11]:
def ig_forward(doc_feats, graph, target_id2):
    graph = graph.to(device)
    return model.explain_forward(doc_feats, graph, target_id2, doc_mask, args.interpret_mode)

bench = Benchmark(ig_forward, tokenizer)
for target_id1, ev in zip(range(test_mask.sum()), evals):
    target_id2 = test_mask.nonzero()[0][target_id1]
    explainers = [IntegratedGradientExplainer(partial(ig_forward, graph=graph, target_id2=target_id2), tokenizer)]
    break
bench.show_evaluation_table(evals[0])

,aopc_compr,aopc_suff,taucorr_loo
Integrated Gradient (x Input),0.24,-0.07,0.92


In [16]:
compr, suff, loo = list(), list(), list()
for ev in evals:
    ig_scores = ev[0].evaluation_scores
    compr.append(ig_scores[0].score)
    suff.append(ig_scores[1].score)
    loo.append(ig_scores[2].score)
np.mean(compr), np.mean(suff), np.mean(loo)

(0.16368961, -0.034812074, 0.8984448736927915)

In [22]:
evals[0][0].explanation.scores.shape

(2699,)

In [23]:
evals[0][0].explanation.scores[:10]

array([ 5.20402772e-04, -3.24476174e-04, -2.94988401e-04,  1.29689236e-04,
       -1.65553585e-04, -1.26535645e-04, -3.06313840e-04, -4.84833736e-05,
       -2.51438405e-05, -8.95353959e-05])

In [28]:
(evals[0][0].explanation.scores**2).sum()

0.007109119334501616

In [31]:
evals[0][0].explanation.scores / np.linalg.norm(evals[0][0].explanation.scores)

array([ 0.00617208, -0.00384835, -0.00349862, ..., -0.00039647,
       -0.00254856, -0.00226628])

In [32]:
np.linalg.norm(evals[0][0].explanation.scores)

0.08431559366156191